# Government720

## Split Training of Dataset

### Importing Client Datasets and Labels

In [1]:
# Access Google Drive for Excel file
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Import packages
import pandas as pd
import tensorflow as tf
import numpy as np

In [3]:
# Read in federated client datasets
file_path = '/content/drive/My Drive/G720_SPLIT.xlsx'  # Finalized dataset
sheets = pd.ExcelFile(file_path).sheet_names

# Define split date
split_date = '2021-01-01'

In [4]:
# Import labels from GLOBAL sheet
global_df = pd.read_excel(file_path, sheet_name='GLOBAL')
global_df.set_index('DATE', inplace=True)  # Set DATE as index

# Prepare global labels split
global_train_mask = global_df.index < split_date
global_test_mask = global_df.index >= split_date

# Define y_train and y_test
y_train_global = global_df.loc[global_train_mask, 'GOVT_SATISFACTION'].values
y_test_global = global_df.loc[global_test_mask, 'GOVT_SATISFACTION'].values

# Prepare client feature data
client_data = {}  # Dictionary to store client datasets

In [5]:
# Load in client datasets
for sheet in sheets:
    if sheet == 'GLOBAL':
        continue  # Skip the GLOBAL sheet

    df = pd.read_excel(file_path, sheet_name=sheet)

    # Split based on date
    train_mask = df["DATE"] < split_date
    test_mask = df["DATE"] >= split_date
    train_df = df[train_mask]
    test_df = df[test_mask]

    # Set DATE as index
    train_df.set_index("DATE", inplace=True)
    test_df.set_index("DATE", inplace=True)

    # Use *only features* (no labels inside client sheet)
    X_train = train_df.values
    X_test = test_df.values

    client_data[sheet] = {
        'X_train': X_train,
        'X_test': X_test
    }

### Model Architecture Setup

In [6]:
# Import model packages
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate, LSTM
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math
from tensorflow.keras.layers import Reshape

In [7]:
# Initialize storage for client models, inputs, and outputs
client_models = {}
client_inputs = []
client_outputs = []

# Create model for each client
for client_name, data in client_data.items():
    input_shape = data['X_train'].shape[1]
    client_input = Input(shape=(input_shape,))
    hidden = Dense(32, activation='relu')(client_input)  # client encoder
    client_model = Model(inputs=client_input, outputs=hidden)
    client_models[client_name] = client_model
    client_inputs.append(client_input)
    client_outputs.append(client_model(client_input))

In [8]:
# Server model to take concatenated client hidden outputs
concatenated = Concatenate()(client_outputs)  # Shape: (batch_size, total_embedding_dim)

# Reshape for LSTM (batch_size, time_steps=1, features=total_embedding_dim)
reshaped = Reshape((1, concatenated.shape[-1]))(concatenated)

# LSTM processing
server_hidden = LSTM(64)(reshaped)  # LSTM processes over time steps (1 here)

# Final output
server_output = Dense(1, activation='linear')(server_hidden)

# Compile split model
split_learning_model = Model(inputs=client_inputs, outputs=server_output)
split_learning_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

### Model Training

In [9]:
# Build input list for model
X_train_list = [client_data[client]['X_train'] for client in client_data.keys()]
X_test_list = [client_data[client]['X_test'] for client in client_data.keys()]

In [10]:
# Train split learning model
split_learning_model.fit(
    X_train_list,
    y_train_global,
    epochs=50,
    batch_size=32,
    verbose=1,
    validation_data=(X_test_list, y_test_global)
)

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - loss: 0.0673 - val_loss: 0.0057
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0081 - val_loss: 0.0063
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0070 - val_loss: 0.0059
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0069 - val_loss: 0.0083
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0064 - val_loss: 0.0132
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0055 - val_loss: 0.0097
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0053 - val_loss: 0.0176
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0053 - val_loss: 0.0197
Epoch 9/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0050 - val_loss: 0.0146
Epoch 10/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0045 - val_loss: 0.0119
Epoch 11/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0051 - val_loss: 0.0215
Epoch 12/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0

### Model Evaluation

In [11]:
# Evaluate split learning model
print("\n--- Evaluating Split Learning Federated Model ---")

y_pred = split_learning_model.predict(X_test_list)

mse = mean_squared_error(y_test_global, y_pred)
mae = mean_absolute_error(y_test_global, y_pred)
rmse = math.sqrt(mse)

print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")


--- Evaluating Split Learning Federated Model ---
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step
MSE: 0.0102
MAE: 0.0846
RMSE: 0.1012


In [12]:
# End of code 4/23/25